# SJFI 500 Quality-Value Index — Backtest (Colab)

This notebook reproduces my full backtest in the browser. It clones the repo, runs `run_backtest.py` unmodified, and shows the summary stats, charts and review audit log.

There is no external data to fetch and no API key to set — the 500-security universe is generated synthetically, with the value and quality premia in the data-generating process disclosed in `synthetic.py` and in §1.5 of the Ground Rules.

Run the cells top to bottom. The whole thing takes about a minute.

## 1. Get the project

In [ ]:
REPO_URL = "https://github.com/sasmitha-jayakody/factor-index.git"

import pathlib, subprocess, sys

PROJECT_DIR = pathlib.Path.cwd() / "factor-index"

if PROJECT_DIR.exists():
    print(f"{PROJECT_DIR} already present, skipping clone.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
    print(f"Project ready at {PROJECT_DIR}")

## 2. Install dependencies

`numpy`, `pandas`, `matplotlib` and `scipy` all ship pre-installed on Colab, so this usually just confirms the versions and exits.

In [ ]:
%pip install -q -r factor-index/requirements.txt

## 3. Run the backtest

This runs the project's own `run_backtest.py` unmodified, so it writes `summary_stats.csv`, `index_levels.csv`, `review_log.csv`, `backtest_report.png` and `factor_exposures.png` into `factor-index/outputs/`, exactly as it does on my machine.

In [ ]:
%cd factor-index
%run run_backtest.py

## 4. Results

The index and its cap-weighted parent are computed on the identical engine and the identical eligible universe, so every difference below is attributable to methodology rather than implementation.

In [ ]:
import pandas as pd

stats = pd.read_csv("outputs/summary_stats.csv", index_col=0)
stats

In [ ]:
from IPython.display import Image, display

display(Image(filename="outputs/backtest_report.png"))

Active factor exposure at each review, measured as the weight gap against the cap-weighted parent multiplied by the constituent z-score. I want this positive and stable through time — a tilt that drifts or flips would mean the scoring is being driven by something other than the factors I intended.

In [ ]:
display(Image(filename="outputs/factor_exposures.png"))

The review log is the audit trail: one row per semi-annual review, with the constituent count, two-way turnover and the cut-off date the data was taken from.

In [ ]:
review_log = pd.read_csv("outputs/review_log.csv")
review_log

## 5. Engine invariants

The tests assert the properties that matter most: a split with no economic price move leaves both index variants unchanged, a deletion causes no jump in the level, a dividend lifts the total-return index by exactly the reinvested yield and leaves the price index alone, and capping converges, respects the cap and raises on infeasibility.

In [ ]:
!python -m pytest tests/ -q

## 6. (Optional) Download the outputs

In [ ]:
import shutil

archive = shutil.make_archive("sjfi_outputs", "zip", "outputs")

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print(f"Not running in Colab - the archive is at {archive}")